In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv


In [2]:
import pandas as pd
import os

# Auto-detect path (safe method)
csv_path = None
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if 'Telco-Customer-Churn.csv' in filename:
            csv_path = os.path.join(dirname, filename)
            break

if csv_path:
    df = pd.read_csv(csv_path)
    print(f"✅ Dataset loaded from: {csv_path}")
    print(f"Shape: {df.shape}")
    df.head()
else:
    print("❌ CSV not found. Please add dataset via Right Panel → + Add Input")

✅ Dataset loaded from: /kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv
Shape: (7043, 21)


In [3]:
def clean_churn_data(df):
    df = df.copy()
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
    df = df.dropna(subset=["TotalCharges", "MonthlyCharges"])
    df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
    df = df.drop(columns=["customerID"])
    return df.reset_index(drop=True)

df_clean = clean_churn_data(df)
print(f"✅ Cleaned shape: {df_clean.shape}")

✅ Cleaned shape: (7032, 20)


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, recall_score
import numpy as np

X = df_clean.drop("Churn", axis=1)
y = df_clean["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Manual preprocessing (no ColumnTransformer)
num_cols = X.select_dtypes("number").columns
cat_cols = X.select_dtypes("object").columns

scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train[num_cols])
X_test_num = scaler.transform(X_test[num_cols])

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_train_cat = ohe.fit_transform(X_train[cat_cols])
X_test_cat = ohe.transform(X_test[cat_cols])

X_train_p = np.hstack([X_train_num, X_train_cat])
X_test_p = np.hstack([X_test_num, X_test_cat])

# Train baseline model
model = RandomForestClassifier(
    n_estimators=100, max_depth=10, random_state=42, 
    class_weight="balanced", n_jobs=-1
)
model.fit(X_train_p, y_train)

# Evaluate
y_pred = model.predict(X_test_p)
y_prob = model.predict_proba(X_test_p)[:, 1]
print(f"📊 Baseline ROC-AUC: {roc_auc_score(y_test, y_prob):.3f}")
print(f"📊 Baseline Recall: {recall_score(y_test, y_pred):.3f}")

📊 Baseline ROC-AUC: 0.832
📊 Baseline Recall: 0.717


In [5]:
# !pip install optuna -q  # Uncomment if not installed
import optuna

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 200),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "class_weight": "balanced",
        "random_state": 42,
        "n_jobs": -1
    }
    
    m = RandomForestClassifier(**params)
    m.fit(X_train_p, y_train)
    
    y_p = m.predict(X_test_p)
    y_prob = m.predict_proba(X_test_p)[:, 1]
    return roc_auc_score(y_test, y_prob)

print("🔍 Running Optuna (5 trials for speed)...")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=5, show_progress_bar=True)

print(f"🏆 Best ROC-AUC: {study.best_value:.3f}")
print(f"🎯 Best Params: {study.best_params}")
best_params = study.best_params

[I 2026-05-16 11:11:20,714] A new study created in memory with name: no-name-9e57f806-a74a-4917-aa09-25b31bb08fe3


🔍 Running Optuna (5 trials for speed)...


  0%|          | 0/5 [00:00<?, ?it/s]

[I 2026-05-16 11:11:21,078] Trial 0 finished with value: 0.8196882554834837 and parameters: {'n_estimators': 57, 'max_depth': 13, 'min_samples_split': 7}. Best is trial 0 with value: 0.8196882554834837.
[I 2026-05-16 11:11:21,735] Trial 1 finished with value: 0.8338622256964037 and parameters: {'n_estimators': 151, 'max_depth': 6, 'min_samples_split': 5}. Best is trial 1 with value: 0.8338622256964037.
[I 2026-05-16 11:11:22,767] Trial 2 finished with value: 0.8227179545583965 and parameters: {'n_estimators': 198, 'max_depth': 13, 'min_samples_split': 3}. Best is trial 1 with value: 0.8338622256964037.
[I 2026-05-16 11:11:23,162] Trial 3 finished with value: 0.832674159164678 and parameters: {'n_estimators': 88, 'max_depth': 5, 'min_samples_split': 6}. Best is trial 1 with value: 0.8338622256964037.
[I 2026-05-16 11:11:24,082] Trial 4 finished with value: 0.8270133715723377 and parameters: {'n_estimators': 183, 'max_depth': 14, 'min_samples_split': 9}. Best is trial 1 with value: 0.833

In [6]:
import joblib
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, recall_score
import numpy as np

# Manual preprocessing (NO ColumnTransformer)
num_cols = X.select_dtypes("number").columns
cat_cols = X.select_dtypes("object").columns

scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train[num_cols])
X_test_num = scaler.transform(X_test[num_cols])

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_train_cat = ohe.fit_transform(X_train[cat_cols])
X_test_cat = ohe.transform(X_test[cat_cols])

X_train_p = np.hstack([X_train_num, X_train_cat])
X_test_p = np.hstack([X_test_num, X_test_cat])

# Use Optuna best params
best_params = {'n_estimators': 166, 'max_depth': 7, 'min_samples_split': 6}
best_params.update({"class_weight": "balanced", "random_state": 42, "n_jobs": -1})

model = RandomForestClassifier(**best_params)
model.fit(X_train_p, y_train)

# Final eval
y_pred = model.predict(X_test_p)
y_prob = model.predict_proba(X_test_p)[:, 1]
print(f"✅ Final ROC-AUC: {roc_auc_score(y_test, y_prob):.3f}")

# 💾 Save with SAME filename but compatible content
joblib.dump({
    "model": model,
    "scaler": scaler,
    "ohe": ohe,
    "num_cols": list(num_cols),
    "cat_cols": list(cat_cols),
    "feature_names": list(X.columns)
}, "/kaggle/working/churn_rf_v1_optuna.pkl", protocol=2, compress=3)

print("✅ Saved as churn_rf_v1_optuna.pkl (compatible version)")

✅ Final ROC-AUC: 0.836
✅ Saved as churn_rf_v1_optuna.pkl (compatible version)
